#Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import trim, col, min, max, count, when

#Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

# Silver Transformations

In [0]:
df.select(F.col("prd_cost")) \
  .summary("count", "min", "25%", "50%", "75%", "max") \
  .show()

#Display the row where value is 2171 in the column prd_cost
df.where(F.col("prd_cost") == 2171).show()

df.groupBy(F.col("prd_key")) \
  .count() \
  .orderBy(F.col("count").desc()) \
  .limit(100).show() #We have sls order number from 1 to 8


In [0]:
invalid_dates_df = df.filter(F.col("prd_end_dt") < F.col("prd_start_dt"))

display(invalid_dates_df)

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Product Key Parsing
From the sales details silver layer we discover the prd key didnt have the same format so we solve this by parsing the key into cat_id and prd_key which is the same in sales details

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

## Normalizing prd_line

In [0]:
df.groupBy(F.col("prd_line")) \
  .count() \
  .orderBy(F.col("count").desc()) \
  .show()

In [0]:
df = (
    df
    .withColumn(
        "prd_line",
        F.when(F.upper(col("prd_line")) == "M", "Mountain")
         .when(F.upper(col("prd_line")) == "R", "Road")
         .when(F.upper(col("prd_line")) == "S", "Other Sales")
         .when(F.upper(col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

## Replacing Null values for 0 in Cost column 
(Comment for the moment as I dont think it would be appropiate for the business logic)

In [0]:
#df = df.withColumn("prd_cost", F.coalesce(F.col("prd_cost"), F.lit(0)))

## Date cast

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

## Renaming Columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_key",
    "prd_cost": "product_cost",
    "prd_nm": "product_name",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanitary checks Dataframe

In [0]:
df.limit(100).display()

# Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

## Sanitary check Silver Table

In [0]:
%sql
SELECT * FROM silver.crm_products LIMIT 10